# ModelLog Anatomy

Audits the ModelLog public manifest against a live tiny-model capture.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.log_forward_pass(
    model,
    x,
    layers_to_save="all",
    save_gradients=True,
    save_function_args=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].activation.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.passes.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.passes.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.log_forward_pass(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.log_forward_pass(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "ModelLog": log,
    "LayerLog": layer_log,
    "LayerPassLog": layer_pass,
    "ModuleLog": module_log,
    "ModulePassLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnPassLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer passes, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## ModelLog Anatomy: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.ModelLog",
    "tl.ModelLog.DEFAULT_FILL_STATE",
    "tl.ModelLog.FORK_POLICY",
    "tl.ModelLog.PORTABLE_STATE_SPEC",
    "tl.ModelLog._activation_hash_cache",
    "tl.ModelLog._activation_recipe_revision",
    "tl.ModelLog._all_layers_logged",
    "tl.ModelLog._all_layers_saved",
    "tl.ModelLog._append_sequence_id",
    "tl.ModelLog._final_to_raw_layer_labels",
    "tl.ModelLog._gradient_layer_nums_to_save",
    "tl.ModelLog._has_direct_writes",
    "tl.ModelLog._intervention_spec",
    "tl.ModelLog._last_hook_handle_ids",
    "tl.ModelLog._layer_counter",
    "tl.ModelLog._layer_num_to_lookup_keys_dict",
    "tl.ModelLog._layer_nums_to_save",
    "tl.ModelLog._layers_where_internal_branches_merge_with_input",
    "tl.ModelLog._lookup_keys_to_layer_num_dict",
    "tl.ModelLog._pass_finished",
    "tl.ModelLog._raw_layer_dict",
    "tl.ModelLog._raw_layer_labels_list",
    "tl.ModelLog._raw_layer_type_counter",
    "tl.ModelLog._raw_to_final_layer_labels",
    "tl.ModelLog._source_code_blob",
    "tl.ModelLog._source_model_ref",
    "tl.ModelLog._spec_revision",
    "tl.ModelLog._unsaved_layers_lookup_keys",
    "tl.ModelLog._warned_direct_write",
    "tl.ModelLog._warned_mutate_in_place",
    "tl.ModelLog.activation_postfunc",
    "tl.ModelLog.activation_postfunc_repr",
    "tl.ModelLog.activation_transform",
    "tl.ModelLog.add_node_overlay",
    "tl.ModelLog.animate_passes",
    "tl.ModelLog.append_run_state_from",
    "tl.ModelLog.attach_hooks",
    "tl.ModelLog.backward_memory_backend",
    "tl.ModelLog.backward_num_passes",
    "tl.ModelLog.backward_peak_memory_bytes",
    "tl.ModelLog.backward_root_grad_fn_id",
    "tl.ModelLog.buffer_layers",
    "tl.ModelLog.buffer_num_passes",
    "tl.ModelLog.buffers",
    "tl.ModelLog.capture_cache_hit",
    "tl.ModelLog.capture_cache_key",
    "tl.ModelLog.capture_cache_path",
    "tl.ModelLog.capture_full_args",
    "tl.ModelLog.capture_kpis",
    "tl.ModelLog.check_metadata_invariants",
    "tl.ModelLog.cleanup",
    "tl.ModelLog.clear_hooks",
    "tl.ModelLog.conditional_arm_edges",
    "tl.ModelLog.conditional_branch_edges",
    "tl.ModelLog.conditional_edge_passes",
    "tl.ModelLog.conditional_elif_edges",
    "tl.ModelLog.conditional_else_edges",
    "tl.ModelLog.conditional_events",
    "tl.ModelLog.conditional_then_edges",
    "tl.ModelLog.current_function_call_barcode",
    "tl.ModelLog.detach_hooks",
    "tl.ModelLog.detach_saved_tensors",
    "tl.ModelLog.detect_loops",
    "tl.ModelLog.do",
    "tl.ModelLog.emit_nvtx",
    "tl.ModelLog.equivalent_operations",
    "tl.ModelLog.find_layers",
    "tl.ModelLog.find_sites",
    "tl.ModelLog.first_nonfinite",
    "tl.ModelLog.flops_by_type",
    "tl.ModelLog.fork",
    "tl.ModelLog.forward_lineno",
    "tl.ModelLog.grad_fn_logs",
    "tl.ModelLog.grad_fn_order",
    "tl.ModelLog.grad_fns",
    "tl.ModelLog.gradient_postfunc",
    "tl.ModelLog.gradient_postfunc_repr",
    "tl.ModelLog.gradients_to_save",
    "tl.ModelLog.graph_shape_hash",
    "tl.ModelLog.has_backward_log",
    "tl.ModelLog.has_conditional_branching",
    "tl.ModelLog.has_gradients",
    "tl.ModelLog.input_id_at_capture",
    "tl.ModelLog.input_layers",
    "tl.ModelLog.input_metadata",
    "tl.ModelLog.input_shape_hash",
    "tl.ModelLog.internally_initialized_layers",
    "tl.ModelLog.internally_terminated_bool_layers",
    "tl.ModelLog.internally_terminated_layers",
    "tl.ModelLog.intervention_ready",
    "tl.ModelLog.intervention_spec",
    "tl.ModelLog.io_format_version",
    "tl.ModelLog.is_appended",
    "tl.ModelLog.is_branching",
    "tl.ModelLog.is_recurrent",
    "tl.ModelLog.keep_unsaved_layers",
    "tl.ModelLog.last_run_ctx",
    "tl.ModelLog.last_run_records",
    "tl.ModelLog.layer_dict_all_keys",
    "tl.ModelLog.layer_dict_main_keys",
    "tl.ModelLog.layer_labels",
    "tl.ModelLog.layer_labels_no_pass",
    "tl.ModelLog.layer_labels_w_pass",
    "tl.ModelLog.layer_list",
    "tl.ModelLog.layer_logs",
    "tl.ModelLog.layer_num_passes",
    "tl.ModelLog.layers",
    "tl.ModelLog.layers_with_params",
    "tl.ModelLog.layers_with_saved_activations",
    "tl.ModelLog.layers_with_saved_gradients",
    "tl.ModelLog.load",
    "tl.ModelLog.log_backward",
    "tl.ModelLog.logging_mode",
    "tl.ModelLog.macs_by_type",
    "tl.ModelLog.manual_tensor_connections",
    "tl.ModelLog.mark_input_output_distances",
    "tl.ModelLog.max_recurrent_loops",
    "tl.ModelLog.model_name",
    "tl.ModelLog.module_filter_fn",
    "tl.ModelLog.modules",
    "tl.ModelLog.name",
    "tl.ModelLog.num_context_lines",
    "tl.ModelLog.num_grad_fns",
    "tl.ModelLog.num_intervening_grad_fns",
    "tl.ModelLog.num_operations",
    "tl.ModelLog.num_streamed_passes",
    "tl.ModelLog.num_tensors_saved",
    "tl.ModelLog.num_tensors_total",
    "tl.ModelLog.observer_spans",
    "tl.ModelLog.operation_history",
    "tl.ModelLog.orphan_layers",
    "tl.ModelLog.output_device",
    "tl.ModelLog.output_layers",
    "tl.ModelLog.param_logs",
    "tl.ModelLog.params",
    "tl.ModelLog.parent_run",
    "tl.ModelLog.pass_end_time",
    "tl.ModelLog.pass_start_time",
    "tl.ModelLog.preview_fastlog",
    "tl.ModelLog.raise_on_nan",
    "tl.ModelLog.random_seed_used",
    "tl.ModelLog.recording_backward",
    "tl.ModelLog.recording_kept",
    "tl.ModelLog.relationship_evidence",
    "tl.ModelLog.release_param_refs",
    "tl.ModelLog.remove",
    "tl.ModelLog.render_dagua_graph",
    "tl.ModelLog.render_graph",
    "tl.ModelLog.replace_run_state_from",
    "tl.ModelLog.replay",
    "tl.ModelLog.replay_from",
    "tl.ModelLog.report_values",
    "tl.ModelLog.rerun",
    "tl.ModelLog.resolve_sites",
    "tl.ModelLog.root_module",
    "tl.ModelLog.run_state",
    "tl.ModelLog.save",
    "tl.ModelLog.save_function_args",
    "tl.ModelLog.save_gradients",
    "tl.ModelLog.save_intervention",
    "tl.ModelLog.save_new_activations",
    "tl.ModelLog.save_raw_activation",
    "tl.ModelLog.save_raw_gradient",
    "tl.ModelLog.save_rng_states",
    "tl.ModelLog.save_source_context",
    "tl.ModelLog.saved_activation_memory",
    "tl.ModelLog.saved_activation_memory_str",
    "tl.ModelLog.set",
    "tl.ModelLog.show",
    "tl.ModelLog.show_backward_graph",
    "tl.ModelLog.source_model_class",
    "tl.ModelLog.source_model_id",
    "tl.ModelLog.streaming_pass_logs",
    "tl.ModelLog.suggest",
    "tl.ModelLog.summary",
    "tl.ModelLog.time_cleanup",
    "tl.ModelLog.time_forward_pass",
    "tl.ModelLog.time_function_calls",
    "tl.ModelLog.time_logging",
    "tl.ModelLog.time_setup",
    "tl.ModelLog.time_total",
    "tl.ModelLog.to_csv",
    "tl.ModelLog.to_dagua_graph",
    "tl.ModelLog.to_json",
    "tl.ModelLog.to_pandas",
    "tl.ModelLog.to_parquet",
    "tl.ModelLog.total_activation_memory",
    "tl.ModelLog.total_activation_memory_str",
    "tl.ModelLog.total_autograd_saved_bytes",
    "tl.ModelLog.total_flops",
    "tl.ModelLog.total_flops_backward",
    "tl.ModelLog.total_flops_forward",
    "tl.ModelLog.total_macs",
    "tl.ModelLog.total_macs_backward",
    "tl.ModelLog.total_macs_forward",
    "tl.ModelLog.total_param_layers",
    "tl.ModelLog.total_param_tensors",
    "tl.ModelLog.total_params",
    "tl.ModelLog.total_params_frozen",
    "tl.ModelLog.total_params_memory",
    "tl.ModelLog.total_params_memory_str",
    "tl.ModelLog.total_params_trainable",
    "tl.ModelLog.train_mode",
    "tl.ModelLog.uncalled_modules",
    "tl.ModelLog.unlogged_layers",
    "tl.ModelLog.unsupported_ops",
    "tl.ModelLog.validate_forward_pass",
    "tl.ModelLog.validate_saved_activations",
    "tl.ModelLog.verbose",
    "tl.ModelLog.visualization_field_audit",
    "tl.ModelLog.weight_fingerprint_at_capture",
    "tl.ModelLog.weight_fingerprint_full",
]

pretty_print_fields(
    log,
    [
        "model_name",
        "model_class_str",
        "num_layers",
        "num_operations",
        "num_passes_run",
        "time_total",
    ],
)
print("contains linear", "linear_1_1" in log)
print("iter first", next(iter(log)).layer_label)

## Identity and timing

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ModelLog",
    "tl.ModelLog.DEFAULT_FILL_STATE",
    "tl.ModelLog.FORK_POLICY",
    "tl.ModelLog.PORTABLE_STATE_SPEC",
    "tl.ModelLog._activation_hash_cache",
    "tl.ModelLog._activation_recipe_revision",
    "tl.ModelLog._all_layers_logged",
    "tl.ModelLog._all_layers_saved",
    "tl.ModelLog._append_sequence_id",
    "tl.ModelLog._final_to_raw_layer_labels",
    "tl.ModelLog._gradient_layer_nums_to_save",
    "tl.ModelLog._has_direct_writes",
    "tl.ModelLog._intervention_spec",
    "tl.ModelLog._last_hook_handle_ids",
    "tl.ModelLog._layer_counter",
    "tl.ModelLog._layer_num_to_lookup_keys_dict",
    "tl.ModelLog._layer_nums_to_save",
    "tl.ModelLog._layers_where_internal_branches_merge_with_input",
    "tl.ModelLog._lookup_keys_to_layer_num_dict",
    "tl.ModelLog._pass_finished",
    "tl.ModelLog._raw_layer_dict",
    "tl.ModelLog._raw_layer_labels_list",
    "tl.ModelLog._raw_layer_type_counter",
    "tl.ModelLog._raw_to_final_layer_labels",
    "tl.ModelLog._source_code_blob",
    "tl.ModelLog._source_model_ref",
    "tl.ModelLog._spec_revision",
    "tl.ModelLog._unsaved_layers_lookup_keys",
    "tl.ModelLog._warned_direct_write",
    "tl.ModelLog._warned_mutate_in_place",
    "tl.ModelLog.activation_postfunc",
    "tl.ModelLog.activation_postfunc_repr",
    "tl.ModelLog.activation_transform",
    "tl.ModelLog.add_node_overlay",
    "tl.ModelLog.animate_passes",
    "tl.ModelLog.append_run_state_from",
]

audit_touch_items("Identity and timing", ITEMS, AUDIT_CONTEXT)

## Layer and module access

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ModelLog.attach_hooks",
    "tl.ModelLog.backward_memory_backend",
    "tl.ModelLog.backward_num_passes",
    "tl.ModelLog.backward_peak_memory_bytes",
    "tl.ModelLog.backward_root_grad_fn_id",
    "tl.ModelLog.buffer_layers",
    "tl.ModelLog.buffer_num_passes",
    "tl.ModelLog.buffers",
    "tl.ModelLog.capture_cache_hit",
    "tl.ModelLog.capture_cache_key",
    "tl.ModelLog.capture_cache_path",
    "tl.ModelLog.capture_full_args",
    "tl.ModelLog.capture_kpis",
    "tl.ModelLog.check_metadata_invariants",
    "tl.ModelLog.cleanup",
    "tl.ModelLog.clear_hooks",
    "tl.ModelLog.conditional_arm_edges",
    "tl.ModelLog.conditional_branch_edges",
    "tl.ModelLog.conditional_edge_passes",
    "tl.ModelLog.conditional_elif_edges",
    "tl.ModelLog.conditional_else_edges",
    "tl.ModelLog.conditional_events",
    "tl.ModelLog.conditional_then_edges",
    "tl.ModelLog.current_function_call_barcode",
    "tl.ModelLog.detach_hooks",
    "tl.ModelLog.detach_saved_tensors",
    "tl.ModelLog.detect_loops",
    "tl.ModelLog.do",
    "tl.ModelLog.emit_nvtx",
    "tl.ModelLog.equivalent_operations",
    "tl.ModelLog.find_layers",
    "tl.ModelLog.find_sites",
    "tl.ModelLog.first_nonfinite",
    "tl.ModelLog.flops_by_type",
    "tl.ModelLog.fork",
    "tl.ModelLog.forward_lineno",
]

audit_touch_items("Layer and module access", ITEMS, AUDIT_CONTEXT)

## Graph and recurrence metadata

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ModelLog.grad_fn_logs",
    "tl.ModelLog.grad_fn_order",
    "tl.ModelLog.grad_fns",
    "tl.ModelLog.gradient_postfunc",
    "tl.ModelLog.gradient_postfunc_repr",
    "tl.ModelLog.gradients_to_save",
    "tl.ModelLog.graph_shape_hash",
    "tl.ModelLog.has_backward_log",
    "tl.ModelLog.has_conditional_branching",
    "tl.ModelLog.has_gradients",
    "tl.ModelLog.input_id_at_capture",
    "tl.ModelLog.input_layers",
    "tl.ModelLog.input_metadata",
    "tl.ModelLog.input_shape_hash",
    "tl.ModelLog.internally_initialized_layers",
    "tl.ModelLog.internally_terminated_bool_layers",
    "tl.ModelLog.internally_terminated_layers",
    "tl.ModelLog.intervention_ready",
    "tl.ModelLog.intervention_spec",
    "tl.ModelLog.io_format_version",
    "tl.ModelLog.is_appended",
    "tl.ModelLog.is_branching",
    "tl.ModelLog.is_recurrent",
    "tl.ModelLog.keep_unsaved_layers",
    "tl.ModelLog.last_run_ctx",
    "tl.ModelLog.last_run_records",
    "tl.ModelLog.layer_dict_all_keys",
    "tl.ModelLog.layer_dict_main_keys",
    "tl.ModelLog.layer_labels",
    "tl.ModelLog.layer_labels_no_pass",
    "tl.ModelLog.layer_labels_w_pass",
    "tl.ModelLog.layer_list",
    "tl.ModelLog.layer_logs",
    "tl.ModelLog.layer_num_passes",
    "tl.ModelLog.layers",
    "tl.ModelLog.layers_with_params",
]

audit_touch_items("Graph and recurrence metadata", ITEMS, AUDIT_CONTEXT)

## Intervention and replay

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ModelLog.layers_with_saved_activations",
    "tl.ModelLog.layers_with_saved_gradients",
    "tl.ModelLog.load",
    "tl.ModelLog.log_backward",
    "tl.ModelLog.logging_mode",
    "tl.ModelLog.macs_by_type",
    "tl.ModelLog.manual_tensor_connections",
    "tl.ModelLog.mark_input_output_distances",
    "tl.ModelLog.max_recurrent_loops",
    "tl.ModelLog.model_name",
    "tl.ModelLog.module_filter_fn",
    "tl.ModelLog.modules",
    "tl.ModelLog.name",
    "tl.ModelLog.num_context_lines",
    "tl.ModelLog.num_grad_fns",
    "tl.ModelLog.num_intervening_grad_fns",
    "tl.ModelLog.num_operations",
    "tl.ModelLog.num_streamed_passes",
    "tl.ModelLog.num_tensors_saved",
    "tl.ModelLog.num_tensors_total",
    "tl.ModelLog.observer_spans",
    "tl.ModelLog.operation_history",
    "tl.ModelLog.orphan_layers",
    "tl.ModelLog.output_device",
    "tl.ModelLog.output_layers",
    "tl.ModelLog.param_logs",
    "tl.ModelLog.params",
    "tl.ModelLog.parent_run",
    "tl.ModelLog.pass_end_time",
    "tl.ModelLog.pass_start_time",
    "tl.ModelLog.preview_fastlog",
    "tl.ModelLog.raise_on_nan",
    "tl.ModelLog.random_seed_used",
    "tl.ModelLog.recording_backward",
    "tl.ModelLog.recording_kept",
    "tl.ModelLog.relationship_evidence",
]

audit_touch_items("Intervention and replay", ITEMS, AUDIT_CONTEXT)

## Export and validation

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ModelLog.release_param_refs",
    "tl.ModelLog.remove",
    "tl.ModelLog.render_dagua_graph",
    "tl.ModelLog.render_graph",
    "tl.ModelLog.replace_run_state_from",
    "tl.ModelLog.replay",
    "tl.ModelLog.replay_from",
    "tl.ModelLog.report_values",
    "tl.ModelLog.rerun",
    "tl.ModelLog.resolve_sites",
    "tl.ModelLog.root_module",
    "tl.ModelLog.run_state",
    "tl.ModelLog.save",
    "tl.ModelLog.save_function_args",
    "tl.ModelLog.save_gradients",
    "tl.ModelLog.save_intervention",
    "tl.ModelLog.save_new_activations",
    "tl.ModelLog.save_raw_activation",
    "tl.ModelLog.save_raw_gradient",
    "tl.ModelLog.save_rng_states",
    "tl.ModelLog.save_source_context",
    "tl.ModelLog.saved_activation_memory",
    "tl.ModelLog.saved_activation_memory_str",
    "tl.ModelLog.set",
    "tl.ModelLog.show",
    "tl.ModelLog.show_backward_graph",
    "tl.ModelLog.source_model_class",
    "tl.ModelLog.source_model_id",
    "tl.ModelLog.streaming_pass_logs",
    "tl.ModelLog.suggest",
    "tl.ModelLog.summary",
    "tl.ModelLog.time_cleanup",
    "tl.ModelLog.time_forward_pass",
    "tl.ModelLog.time_function_calls",
    "tl.ModelLog.time_logging",
    "tl.ModelLog.time_setup",
]

audit_touch_items("Export and validation", ITEMS, AUDIT_CONTEXT)

## Private state fields

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.ModelLog.time_total",
    "tl.ModelLog.to_csv",
    "tl.ModelLog.to_dagua_graph",
    "tl.ModelLog.to_json",
    "tl.ModelLog.to_pandas",
    "tl.ModelLog.to_parquet",
    "tl.ModelLog.total_activation_memory",
    "tl.ModelLog.total_activation_memory_str",
    "tl.ModelLog.total_autograd_saved_bytes",
    "tl.ModelLog.total_flops",
    "tl.ModelLog.total_flops_backward",
    "tl.ModelLog.total_flops_forward",
    "tl.ModelLog.total_macs",
    "tl.ModelLog.total_macs_backward",
    "tl.ModelLog.total_macs_forward",
    "tl.ModelLog.total_param_layers",
    "tl.ModelLog.total_param_tensors",
    "tl.ModelLog.total_params",
    "tl.ModelLog.total_params_frozen",
    "tl.ModelLog.total_params_memory",
    "tl.ModelLog.total_params_memory_str",
    "tl.ModelLog.total_params_trainable",
    "tl.ModelLog.train_mode",
    "tl.ModelLog.uncalled_modules",
    "tl.ModelLog.unlogged_layers",
    "tl.ModelLog.unsupported_ops",
    "tl.ModelLog.validate_forward_pass",
    "tl.ModelLog.validate_saved_activations",
    "tl.ModelLog.verbose",
    "tl.ModelLog.visualization_field_audit",
    "tl.ModelLog.weight_fingerprint_at_capture",
    "tl.ModelLog.weight_fingerprint_full",
]

audit_touch_items("Private state fields", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.ModelLog",
    "tl.ModelLog.DEFAULT_FILL_STATE",
    "tl.ModelLog.FORK_POLICY",
    "tl.ModelLog.PORTABLE_STATE_SPEC",
    "tl.ModelLog._activation_hash_cache",
    "tl.ModelLog._activation_recipe_revision",
    "tl.ModelLog._all_layers_logged",
    "tl.ModelLog._all_layers_saved",
    "tl.ModelLog._append_sequence_id",
    "tl.ModelLog._final_to_raw_layer_labels",
    "tl.ModelLog._gradient_layer_nums_to_save",
    "tl.ModelLog._has_direct_writes",
    "tl.ModelLog._intervention_spec",
    "tl.ModelLog._last_hook_handle_ids",
    "tl.ModelLog._layer_counter",
    "tl.ModelLog._layer_num_to_lookup_keys_dict",
    "tl.ModelLog._layer_nums_to_save",
    "tl.ModelLog._layers_where_internal_branches_merge_with_input",
    "tl.ModelLog._lookup_keys_to_layer_num_dict",
    "tl.ModelLog._pass_finished",
    "tl.ModelLog._raw_layer_dict",
    "tl.ModelLog._raw_layer_labels_list",
    "tl.ModelLog._raw_layer_type_counter",
    "tl.ModelLog._raw_to_final_layer_labels",
    "tl.ModelLog._source_code_blob",
    "tl.ModelLog._source_model_ref",
    "tl.ModelLog._spec_revision",
    "tl.ModelLog._unsaved_layers_lookup_keys",
    "tl.ModelLog._warned_direct_write",
    "tl.ModelLog._warned_mutate_in_place",
    "tl.ModelLog.activation_postfunc",
    "tl.ModelLog.activation_postfunc_repr",
    "tl.ModelLog.activation_transform",
    "tl.ModelLog.add_node_overlay",
    "tl.ModelLog.animate_passes",
    "tl.ModelLog.append_run_state_from",
    "tl.ModelLog.attach_hooks",
    "tl.ModelLog.backward_memory_backend",
    "tl.ModelLog.backward_num_passes",
    "tl.ModelLog.backward_peak_memory_bytes",
    "tl.ModelLog.backward_root_grad_fn_id",
    "tl.ModelLog.buffer_layers",
    "tl.ModelLog.buffer_num_passes",
    "tl.ModelLog.buffers",
    "tl.ModelLog.capture_cache_hit",
    "tl.ModelLog.capture_cache_key",
    "tl.ModelLog.capture_cache_path",
    "tl.ModelLog.capture_full_args",
    "tl.ModelLog.capture_kpis",
    "tl.ModelLog.check_metadata_invariants",
    "tl.ModelLog.cleanup",
    "tl.ModelLog.clear_hooks",
    "tl.ModelLog.conditional_arm_edges",
    "tl.ModelLog.conditional_branch_edges",
    "tl.ModelLog.conditional_edge_passes",
    "tl.ModelLog.conditional_elif_edges",
    "tl.ModelLog.conditional_else_edges",
    "tl.ModelLog.conditional_events",
    "tl.ModelLog.conditional_then_edges",
    "tl.ModelLog.current_function_call_barcode",
    "tl.ModelLog.detach_hooks",
    "tl.ModelLog.detach_saved_tensors",
    "tl.ModelLog.detect_loops",
    "tl.ModelLog.do",
    "tl.ModelLog.emit_nvtx",
    "tl.ModelLog.equivalent_operations",
    "tl.ModelLog.find_layers",
    "tl.ModelLog.find_sites",
    "tl.ModelLog.first_nonfinite",
    "tl.ModelLog.flops_by_type",
    "tl.ModelLog.fork",
    "tl.ModelLog.forward_lineno",
    "tl.ModelLog.grad_fn_logs",
    "tl.ModelLog.grad_fn_order",
    "tl.ModelLog.grad_fns",
    "tl.ModelLog.gradient_postfunc",
    "tl.ModelLog.gradient_postfunc_repr",
    "tl.ModelLog.gradients_to_save",
    "tl.ModelLog.graph_shape_hash",
    "tl.ModelLog.has_backward_log",
    "tl.ModelLog.has_conditional_branching",
    "tl.ModelLog.has_gradients",
    "tl.ModelLog.input_id_at_capture",
    "tl.ModelLog.input_layers",
    "tl.ModelLog.input_metadata",
    "tl.ModelLog.input_shape_hash",
    "tl.ModelLog.internally_initialized_layers",
    "tl.ModelLog.internally_terminated_bool_layers",
    "tl.ModelLog.internally_terminated_layers",
    "tl.ModelLog.intervention_ready",
    "tl.ModelLog.intervention_spec",
    "tl.ModelLog.io_format_version",
    "tl.ModelLog.is_appended",
    "tl.ModelLog.is_branching",
    "tl.ModelLog.is_recurrent",
    "tl.ModelLog.keep_unsaved_layers",
    "tl.ModelLog.last_run_ctx",
    "tl.ModelLog.last_run_records",
    "tl.ModelLog.layer_dict_all_keys",
    "tl.ModelLog.layer_dict_main_keys",
    "tl.ModelLog.layer_labels",
    "tl.ModelLog.layer_labels_no_pass",
    "tl.ModelLog.layer_labels_w_pass",
    "tl.ModelLog.layer_list",
    "tl.ModelLog.layer_logs",
    "tl.ModelLog.layer_num_passes",
    "tl.ModelLog.layers",
    "tl.ModelLog.layers_with_params",
    "tl.ModelLog.layers_with_saved_activations",
    "tl.ModelLog.layers_with_saved_gradients",
    "tl.ModelLog.load",
    "tl.ModelLog.log_backward",
    "tl.ModelLog.logging_mode",
    "tl.ModelLog.macs_by_type",
    "tl.ModelLog.manual_tensor_connections",
    "tl.ModelLog.mark_input_output_distances",
    "tl.ModelLog.max_recurrent_loops",
    "tl.ModelLog.model_name",
    "tl.ModelLog.module_filter_fn",
    "tl.ModelLog.modules",
    "tl.ModelLog.name",
    "tl.ModelLog.num_context_lines",
    "tl.ModelLog.num_grad_fns",
    "tl.ModelLog.num_intervening_grad_fns",
    "tl.ModelLog.num_operations",
    "tl.ModelLog.num_streamed_passes",
    "tl.ModelLog.num_tensors_saved",
    "tl.ModelLog.num_tensors_total",
    "tl.ModelLog.observer_spans",
    "tl.ModelLog.operation_history",
    "tl.ModelLog.orphan_layers",
    "tl.ModelLog.output_device",
    "tl.ModelLog.output_layers",
    "tl.ModelLog.param_logs",
    "tl.ModelLog.params",
    "tl.ModelLog.parent_run",
    "tl.ModelLog.pass_end_time",
    "tl.ModelLog.pass_start_time",
    "tl.ModelLog.preview_fastlog",
    "tl.ModelLog.raise_on_nan",
    "tl.ModelLog.random_seed_used",
    "tl.ModelLog.recording_backward",
    "tl.ModelLog.recording_kept",
    "tl.ModelLog.relationship_evidence",
    "tl.ModelLog.release_param_refs",
    "tl.ModelLog.remove",
    "tl.ModelLog.render_dagua_graph",
    "tl.ModelLog.render_graph",
    "tl.ModelLog.replace_run_state_from",
    "tl.ModelLog.replay",
    "tl.ModelLog.replay_from",
    "tl.ModelLog.report_values",
    "tl.ModelLog.rerun",
    "tl.ModelLog.resolve_sites",
    "tl.ModelLog.root_module",
    "tl.ModelLog.run_state",
    "tl.ModelLog.save",
    "tl.ModelLog.save_function_args",
    "tl.ModelLog.save_gradients",
    "tl.ModelLog.save_intervention",
    "tl.ModelLog.save_new_activations",
    "tl.ModelLog.save_raw_activation",
    "tl.ModelLog.save_raw_gradient",
    "tl.ModelLog.save_rng_states",
    "tl.ModelLog.save_source_context",
    "tl.ModelLog.saved_activation_memory",
    "tl.ModelLog.saved_activation_memory_str",
    "tl.ModelLog.set",
    "tl.ModelLog.show",
    "tl.ModelLog.show_backward_graph",
    "tl.ModelLog.source_model_class",
    "tl.ModelLog.source_model_id",
    "tl.ModelLog.streaming_pass_logs",
    "tl.ModelLog.suggest",
    "tl.ModelLog.summary",
    "tl.ModelLog.time_cleanup",
    "tl.ModelLog.time_forward_pass",
    "tl.ModelLog.time_function_calls",
    "tl.ModelLog.time_logging",
    "tl.ModelLog.time_setup",
    "tl.ModelLog.time_total",
    "tl.ModelLog.to_csv",
    "tl.ModelLog.to_dagua_graph",
    "tl.ModelLog.to_json",
    "tl.ModelLog.to_pandas",
    "tl.ModelLog.to_parquet",
    "tl.ModelLog.total_activation_memory",
    "tl.ModelLog.total_activation_memory_str",
    "tl.ModelLog.total_autograd_saved_bytes",
    "tl.ModelLog.total_flops",
    "tl.ModelLog.total_flops_backward",
    "tl.ModelLog.total_flops_forward",
    "tl.ModelLog.total_macs",
    "tl.ModelLog.total_macs_backward",
    "tl.ModelLog.total_macs_forward",
    "tl.ModelLog.total_param_layers",
    "tl.ModelLog.total_param_tensors",
    "tl.ModelLog.total_params",
    "tl.ModelLog.total_params_frozen",
    "tl.ModelLog.total_params_memory",
    "tl.ModelLog.total_params_memory_str",
    "tl.ModelLog.total_params_trainable",
    "tl.ModelLog.train_mode",
    "tl.ModelLog.uncalled_modules",
    "tl.ModelLog.unlogged_layers",
    "tl.ModelLog.unsupported_ops",
    "tl.ModelLog.validate_forward_pass",
    "tl.ModelLog.validate_saved_activations",
    "tl.ModelLog.verbose",
    "tl.ModelLog.visualization_field_audit",
    "tl.ModelLog.weight_fingerprint_at_capture",
    "tl.ModelLog.weight_fingerprint_full",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")